# Assignment 2
**Course:** AI-614 – Advanced AI for Unstructured Data and Strategic Analytics


---

**Questions:**
1. Design and Implement Chatbots and Dialogue System (Both Dialogflow and Fine-tuning Techniques)
2. Design and Implement Text Summarization System
3. Design and Implement Question Answering System
4. Design and Implement Named Entity Recognition System

---

Submitted by: Rohit Kumar , Mtech AI, Roll No:- 25901334

# Q2. Text Summarization System
### Using Facebook BART (facebook/bart-large-cnn) for abstractive text summarization

In [ ]:
%pip install transformers

In [ ]:
%pip install ipywidgets

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load model and tokenizer
model_name = "facebook/bart-large-cnn"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def summarize(text, max_len=130, min_len=30):
    inputs = tokenizer(text, return_tensors="pt", truncation=True)

    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=max_len,
        min_length=min_len,
        length_penalty=2.0,
        num_beams=4,
        early_stopping=True
    )

    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)


# Example
text = """Artificial Intelligence is transforming industries by automating tasks and improving efficiency..."""
print(summarize(text))

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

Artificial Intelligence is transforming industries by automating tasks. It is also improving efficiency and reducing the need for humans to do tasks. Here are a few examples of how it can be used in the workplace.


# Q3. Question Answering System
### Using LLaMA-based model (Trama GGUF) for context-aware question answering

In [ ]:
%pip install --upgrade pip setuptools wheel
%pip install llama-cpp-python --prefer-binary

In [ ]:
!pip install llama-cpp-python

In [ ]:
 # !pip install llama-cpp-python

from llama_cpp import Llama

llm = Llama.from_pretrained(
	repo_id="Trina-QwQ/Trama",
	filename="trama.gguf",
)


In [ ]:
# Define context and question
context = "My name is Ohi and I live in Bihar."
question = "What is my name?"

# Create proper prompt
messages = [
    {"role": "system", "content": "Answer the question based on the given context."},
    {"role": "user", "content": f"Context: {context}\nQuestion: {question}"}
]

# Generate answer
response = llm.create_chat_completion(messages=messages)

print(response["choices"][0]["message"]["content"])

# Q4. Named Entity Recognition (NER) System
### Using BERT-base NER model (dslim/bert-base-NER) for entity extraction

In [ ]:
%pip install transformers torch

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

model_name = "dslim/bert-base-NER"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)

In [ ]:
def ner(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True)
    outputs = model(**inputs)

    predictions = torch.argmax(outputs.logits, dim=2)
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    labels = model.config.id2label

    entities = []
    current_entity = ""

    for token, pred in zip(tokens, predictions[0]):
        label = labels[pred.item()]

        if label.startswith("B-"):
            if current_entity:
                entities.append(current_entity)
            current_entity = token.replace("##", "")

        elif label.startswith("I-"):
            current_entity += " " + token.replace("##", "")

        else:
            if current_entity:
                entities.append(current_entity)
                current_entity = ""

    return entities

In [ ]:
text = "Ohi lives in Banglore and works at Amazon."
print(ner(text))

text = "she lives in Kolkata and works at fintech."
print(ner(text))

In [ ]:
def ner_with_labels(text):
    inputs = tokenizer(text, return_tensors="pt")
    outputs = model(**inputs)

    preds = torch.argmax(outputs.logits, dim=2)
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    labels = model.config.id2label

    result = []

    for token, pred in zip(tokens, preds[0]):
        label = labels[pred.item()]
        if label != "O":
            result.append((token, label))

    return result

In [ ]:
text = "Ohi lives in Banglore and works at Amazon."
print(ner(text))

text = "she lives in Kolkata and works at fintech."
print(ner(text))

# Q1. Chatbots and Dialogue System
### Using LLaMA-based model (Trama GGUF) as the fine-tuned conversational backend

In [ ]:
from llama_cpp import Llama

llm = Llama.from_pretrained(
    repo_id="Trina-QwQ/Trama",
    filename="trama.gguf",
)

def chat():
    messages = [
        {"role": "system", "content": "You are a helpful assistant."}
    ]

    while True:
        user_input = input("You: ")
        if user_input.lower() in ["exit", "quit", "bye"]:
            print("Bot: Goodbye! 👋")
            break

        messages.append({"role": "user", "content": user_input})

        response = llm.create_chat_completion(messages=messages)

        reply = response["choices"][0]["message"]["content"]

        print("Bot:", reply)

        messages.append({"role": "assistant", "content": reply})

chat()